# Misleading Variables: Some Columns Are Traps

## Cleanup is not housekeeping — it is deciding what evidence belongs

Run the cell below first. It enlarges the font for both code and markdown so the notebook is easy to read while walking through it in class.

In [ ]:
from IPython.display import HTML, display

display(HTML("""
<style>
/* Rendered markdown */
.jp-RenderedHTMLCommon,
.jp-RenderedMarkdown,
.rendered_html {
    font-size: 24px !important;
    line-height: 1.5 !important;
}
.jp-RenderedHTMLCommon h1, .rendered_html h1 { font-size: 40px !important; }
.jp-RenderedHTMLCommon h2, .rendered_html h2 { font-size: 34px !important; }
.jp-RenderedHTMLCommon h3, .rendered_html h3 { font-size: 30px !important; }
.jp-RenderedHTMLCommon h4, .rendered_html h4 { font-size: 28px !important; }
.jp-RenderedHTMLCommon table, .rendered_html table {
    font-size: 22px !important;
}

/* Code editor (CodeMirror, used by classic + JupyterLab) */
.CodeMirror, .cm-editor, .jp-Editor, .jp-InputArea-editor {
    font-size: 24px !important;
}
.cm-content, .cm-line { font-size: 24px !important; }

/* Code output (print, tracebacks, DataFrame text) */
.jp-OutputArea-output,
.output_area,
.output pre,
.jp-RenderedText pre {
    font-size: 22px !important;
}

/* DataFrame tables in output */
.dataframe, .dataframe th, .dataframe td {
    font-size: 22px !important;
}
</style>
"""))


### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

plt.rcParams['font.size'] = 16
plt.rcParams['figure.figsize'] = (8, 5)

## The story

A churn dataset contains useful predictors (plan, tenure), identifiers (`customer_id`), dates, and variables that *happen after* churn (`refund_after_churn`). The latter look powerfully predictive, but they leak the answer.

## 1. Load the churn dataset

In [ ]:
churn = pd.read_csv("../data/misleading_variables_churn.csv")
churn.head()

In [ ]:
churn.info()

## 2. Column audit with `df.columns`

Ask whether each column is an **identifier**, **outcome**, **predictor**, **date**, **proxy**, or **leak**.

In [ ]:
list(churn.columns)

## 3. Type audit with `df.dtypes`

Types reveal disguised dates, booleans, and numbers.

In [ ]:
churn.dtypes

## 4. Convert with `df.astype()`

Use `astype()` when the intended type is clear.

In [ ]:
churn["churned"] = churn["churned"].astype("bool")
churn["churned"].dtype

## 5. Convert dates with `pd.to_datetime()`

In [ ]:
churn["signup_date"] = pd.to_datetime(churn["signup_date"])
churn["signup_date"].dtype

## 6. Find suspicious correlations

A variable can look powerful because it leaks future information.

In [ ]:
numeric = churn.select_dtypes(include="number")
numeric.corr(numeric_only=True).round(3)

Pay special attention to anything that correlates strongly with `churned`. Ask: *could this value have been known at decision time, or is it a consequence of churn?*

## 7. Cross-check a suspect with the outcome

`refund_after_churn` smells like a leak — by name alone.

In [ ]:
pd.crosstab(churn["churned"], churn["refund_after_churn"])

If refunds only happen *after* churn, this column can perfectly predict the outcome — but only because the outcome already happened.

## 8. Drop columns with `df.drop()`

Dropping columns is an **analytical decision** that should be explained, not a default cleanup step.

In [ ]:
safe = churn.drop(columns=["customer_id", "refund_after_churn", "last_login_days_ago"])
list(safe.columns)

Why each drop:

- `customer_id` — identifier, no predictive content
- `refund_after_churn` — happens after the outcome (**leak**)
- `last_login_days_ago` — measured at extraction time, may also leak

## 9. Drop rows vs. drop columns

Same verb, very different consequence.

In [ ]:
print("Drop rows with any NA:  ", churn.dropna().shape)
print("Drop one column:        ", churn.drop(columns=["customer_id"]).shape)

## Mini-lab: leakage hunt

In [ ]:
print(churn.columns.tolist())
print(churn.dtypes)
safe = churn.drop(columns=["customer_id", "refund_after_churn"])
print("Kept columns:", list(safe.columns))

## Discussion

- For each remaining column, when in the customer's lifetime is its value known? Before churn, at churn, or after?
- Which columns would you keep for an honest churn-prediction EDA, and which would you justify dropping in writing?

## Takeaway

Functions introduced / reinforced: `columns`, `dtypes`, `astype`, `to_datetime`, `select_dtypes`, `corr`, `drop`.

**Concept learned: not every column deserves to survive EDA.**